In [ ]:
# Cluster Analysis - Language Learning App
# The business goal is to determine the best markets to enter with AI-powered
# language learning app targeting professionals who learn English for work.
# I have collected the data from AppMagic on the median ranking of key representative language learning apps
# from 03/2023 to 03/2026. The first is the ranking of downloads among free apps,
# the second is the ranking by revenue.
# I am going to perform k-means cluster analysis to identify how the countries group
# across these two dimensions.

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from scipy.stats import kruskal, spearmanr
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker
from scipy import stats

# ==========================================
# 1. LOAD AND PREPARE APP RANKING DATA
# ==========================================
df_apps = pd.read_csv('/content/drive/MyDrive/Datasets_DA/EngApps_v2.csv', sep=';')
df_head = df_apps.head()
print(df_head)

# ISOLATE THE APP NAMES
# We need a list of just the app names to tell the algorithm what to melt.
# We need to skip the first 2 columns as these are not app names.
app_cols = df_apps.columns[2:].tolist()

# RESHAPE STEP 1: MELT (Wide to Long Format)
# K-Means cannot read data stretched horizontally across multiple app columns.

melted = df_apps.melt(
    id_vars=['Country', 'Ranking'],
    var_name='App',
    value_name='Rank'
)

# RESHAPE STEP 2: PIVOT (Aligning Free and Gross Ranks)
# Right now, Free Rank and Gross Rank are on separate rows. We need them side-by-side
# so we can calculate the math between them.
pivoted = melted.pivot_table(
    index=['Country', 'App'],
    columns='Ranking',
    values='Rank'
).reset_index()

# CLEAN THE NEW TABLE
# The pivot table gave our columns awkward, long names. We rename them for clean code.
pivoted.rename(columns={
    'Top Free Median': 'Free_Rank',
    'Top Grossing Median': 'Gross_Rank'
}, inplace=True)

# DROP MISSING DATA
# If an app didn't rank in a specific country, it shows up as NaN (Not a Number).
# Machine learning algorithms will crash if they encounter a NaN.
# 'dropna' permanently deletes any row missing a Free or Gross rank.
pivoted = pivoted.dropna(subset=['Free_Rank', 'Gross_Rank'])



# ==========================================
# 2. CLUSTERING
# ==========================================

# APPLY MATHEMATICAL CORRECTIONS (Log Transformation)
# App store ranks follow a power-law distribution. The difference in market size
# between rank #1 and #5 is massive, but the difference between #500 and #505 is tiny.
# Applying a logarithm ensures earlier ranks weight more than later.
# We add + 1 to the rank just in case a rank of 0 somehow exists.
pivoted['Log_Free_Rank'] = np.log10(pivoted['Free_Rank'] + 1)
pivoted['Log_Gross_Rank'] = np.log10(pivoted['Gross_Rank'] + 1)

# AGGREGATE
# We no longer care about individual apps. We want the overall market behavior.
# We group all the rows by 'Country' and calculate the median of their log-transformed ranks.
# This leaves us with exactly 45 rows (one for each country) and two behavioral metrics.
country_stats = pivoted.groupby('Country').agg({
    'Log_Free_Rank': 'median',  # Overall demand for downloading the apps
    'Log_Gross_Rank': 'median'  # Overall willingness to pay for the apps
}).reset_index()

# Print the final cleaned table to verify it worked before clustering
print(country_stats.head())

# ISOLATE THE FEATURES
# We isolate the two behavioral metrics we just calculated.
features = ['Log_Free_Rank', 'Log_Gross_Rank']
X = country_stats[features]

# STANDARDIZE THE DATA
# K-Means calculates physical geometric distance between points.
# If Gross Rank has a naturally wider variance than Free Rank across our 30 countries,
# the algorithm will treat Gross Rank as "more important" simply because the numbers are spread out.
# StandardScaler converts all numbers into Z-scores (forcing the variance of both to exactly 1.0).
# This guarantees that Usage and Monetization get an equal 50/50 vote in the clustering process.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


#Next, we need to determine the number of clusters

# INITIALIZE TRACKING LISTS FOR THE TEST
wcss = []              # Within-Cluster Sum of Squares (for the Elbow Method)
K_range = range(2, 11) # We test every scenario from 2 clusters up to 10 clusters

# RUN THE SIMULATIONS
for k in K_range:
    # Initialize the K-Means algorithm for the current 'k'
    # random_state=42 ensures our results are exactly reproducible every time we run the script
    # n_init=10 tells the algorithm to run 10 times with different random starting points and pick the best one
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)

    # Record the WCSS (Inertia measures how far points are from their cluster center)
    wcss.append(kmeans.inertia_)


# PLOT THE RESULTS
plt.figure(figsize=(8, 5))
plt.subplot(1, 2, 1)
plt.plot(K_range, wcss, marker='o', linestyle='-', color='b')
plt.title('Elbow Method (WCSS)')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Within-Cluster Sum of Squares')
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.savefig('cluster_validation.png')
print("\nValidation chart saved as 'cluster_validation.png'")

#The exact scores
print("\nK-Means Validation Scores:")
for k, w in zip(K_range, wcss):
    print(f"{k}\t{w:.2f}")


#The test shows the 4 is the number of clusters we should go with.
#Moving on to the clustering itself.
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
# Add + 1 to shift the zero-indexed labels (0, 1, 2, 3) to business-friendly labels (1, 2, 3, 4)
country_stats['Cluster'] = kmeans.fit_predict(X_scaled) + 1


# Visualisation
plt.figure(figsize=(12, 10))
sns.set_theme(style="whitegrid")

# Generate the scatter plot
# We map Usage (Free Rank) to the X-axis and Monetization (Gross Rank) to the Y-axis.
# 'hue' automatically color-codes the dots based on the 4 K-Means clusters.
# 'palette="Set1"' gives distinct, professional colors.
sns.scatterplot(
    data=country_stats,
    x='Log_Free_Rank',
    y='Log_Gross_Rank',
    hue='Cluster',
    palette='Set1',
    s=120,          # Size of the dots
    alpha=0.8       # Slight transparency to see overlapping dots
)

# I invert the axes
# App store ranks go as follows: lower is better.
# By inverting both axes, the "Top Right" quadrant now represents the markets
# with the highest download volume AND the highest revenue.
plt.gca().invert_xaxis()
plt.gca().invert_yaxis()

# Add Country Labels to the dots
for i in range(country_stats.shape[0]):
    plt.text(
        x=country_stats['Log_Free_Rank'].iloc[i] + 0.02, # Offset slightly right
        y=country_stats['Log_Gross_Rank'].iloc[i],
        s=country_stats['Country'].iloc[i],
        fontsize=9,
        color='black'
    )

# Clean up the labels and titles
plt.title('Global English Learning App Market: Usage vs. Monetization', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('HIGHER USAGE ➔', fontsize=12, labelpad=10)
plt.ylabel('HIGHER REVENUE ➔', fontsize=12, labelpad=10)

# Move the legend outside the chart so it doesn't cover any data points
plt.legend(title='Market Cluster', bbox_to_anchor=(1.05, 1), loc='upper left')

# Save the visualization to your folder
plt.tight_layout()
plt.savefig('market_cluster_matrix.png', dpi=300)
print("\nSuccess: Saved 'market_cluster_matrix.png' to your directory.")
plt.show()

# ==========================================
# 3. LOAD AND MERGE MACROECONOMIC DATA
# ==========================================

# 3A. LOAD EF EPI DATA
df_epi = pd.read_csv('/content/drive/MyDrive/Datasets_DA/EPI_v2.csv', sep=';')
df_epi.rename(columns={'EF_EPI_25': 'EF_EPI_Score'}, inplace=True)

# 3B. LOAD GDP DATA
# The file has 3 rows of metadata at the top. We must use skiprows=3.
# It also uses semicolons as delimiters.
df_gdp = pd.read_csv('/content/drive/MyDrive/Datasets_DA/GDP_PPP.csv', sep=';', skiprows=3)

# 3C. CLEAN GDP DATA
# We isolate just the country name and the 2024 GDP column.
# We rename them so they merge cleanly with the existing datasets.
df_gdp_clean = df_gdp[['Country Name', '2024']].copy()
df_gdp_clean.rename(columns={'Country Name': 'Country', '2024': 'GDP_PPP'}, inplace=True)
# Target and replace only the UAE in the GDP dataframe
df_gdp_clean['Country'] = df_gdp_clean['Country'].replace({'United Arab Emirates': 'UAE'})

# 3D. FIX THE EUROPEAN DECIMALS
# The World Bank uses commas instead of dots for decimals (e.g., 51262,51).
# We must convert these to strings, replace the comma with a dot, and turn them
# back into numeric floats so the test can perform math on them.
df_gdp_clean['GDP_PPP'] = df_gdp_clean['GDP_PPP'].astype(str).str.replace(',', '.')
df_gdp_clean['GDP_PPP'] = pd.to_numeric(df_gdp_clean['GDP_PPP'], errors='coerce')

# 3E. EXECUTE THE FINAL MERGE
# We merge the app clustering stats, the EPI scores, and the GDP data together.
# 'how=inner' ensures that if a country is missing from ANY of the three lists,
# it gets safely dropped from the final analysis to prevent crashes.
final_df = country_stats.merge(df_epi, on='Country', how='inner')
final_df = final_df.merge(df_gdp_clean, on='Country', how='inner')

# Verify the merge was successful (Should print 43 countries). There are 45 countries in total,
# but EPI is missing for Taiwan and Singapore.
print(f"\nTotal countries successfully merged for macro-analysis: {len(final_df)}")
print(final_df[['Country', 'Cluster', 'EF_EPI_Score', 'GDP_PPP']].head())

# ==========================================
# 4. STATISTICAL TESTING (NON-PARAMETRIC)
# ==========================================

print("--- CLUSTER PROFILES (the Kruskal-Wallis direction) ---")
# Calculate the median GDP and EPI for each cluster to see the actual direction
cluster_profiles = final_df.groupby('Cluster').agg({
    'GDP_PPP': 'median',
    'EF_EPI_Score': 'median',
    'Log_Free_Rank': 'median',
    'Log_Gross_Rank': 'median'
}).round(2)
print(cluster_profiles)
print("\n")

print("--- KRUSKAL-WALLIS H-TEST ---")
# Does GDP differ significantly between the 4 clusters?
gdp_groups = [group['GDP_PPP'].values for name, group in final_df.groupby('Cluster')]
stat_gdp, p_gdp = kruskal(*gdp_groups)
print(f"GDP across Clusters: H-Stat = {stat_gdp:.2f}, p-value = {p_gdp:.4f}")

# Does EF EPI differ significantly between the 4 clusters?
# The .dropna() specifically removes the two missing countries from this array only
epi_groups = [group['EF_EPI_Score'].dropna().values for name, group in final_df.groupby('Cluster')]
stat_epi, p_epi = kruskal(*epi_groups)
print(f"EF EPI across Clusters: H-Stat = {stat_epi:.2f}, p-value = {p_epi:.4f}\n")



# ==========================================
# 5. VISUALIZATIONS
# ==========================================
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Boxplot: GDP by Cluster
# Added hue='Cluster' and legend=False to comply with Seaborn's strict new API
sns.boxplot(ax=axes[0], x='Cluster', y='GDP_PPP', data=final_df,
            hue='Cluster', palette='Set1', boxprops={'alpha': 0.7}, legend=False)
sns.stripplot(ax=axes[0], x='Cluster', y='GDP_PPP', data=final_df,
              color='black', alpha=0.6, jitter=True)

axes[0].set_title('Macro Wealth: GDP per Capita (PPP) by Cluster', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('Market Cluster', fontsize=12)
axes[0].set_ylabel('GDP (PPP)', fontsize=12)

# Format the Y-axis to look like currency (e.g., $50,000) instead of raw floats
formatter = ticker.StrMethodFormatter('${x:,.0f}')
axes[0].yaxis.set_major_formatter(formatter)

# Boxplot: EF EPI by Cluster
# Added hue='Cluster' and legend=False here as well
sns.boxplot(ax=axes[1], x='Cluster', y='EF_EPI_Score', data=final_df,
            hue='Cluster', palette='Set1', boxprops={'alpha': 0.7}, legend=False)
sns.stripplot(ax=axes[1], x='Cluster', y='EF_EPI_Score', data=final_df,
              color='black', alpha=0.6, jitter=True)

axes[1].set_title('Linguistic Friction: EF EPI Score by Cluster', fontsize=14, fontweight='bold', pad=15)
axes[1].set_xlabel('Market Cluster', fontsize=12)
axes[1].set_ylabel('EF EPI Score', fontsize=12)

plt.tight_layout()
plt.savefig('cluster_macro_analysis.png', dpi=300)
print("Success: Final visualization saved as 'cluster_macro_analysis.png'")


# ==========================================
# 6. App Mix
# ==========================================
df_apps = pd.read_csv('/content/drive/MyDrive/Datasets_DA/EngApps_v2.csv', sep=';')

melted = df_apps.melt(id_vars=['Country', 'Ranking'], var_name='App', value_name='Rank')
melted['Rank'] = pd.to_numeric(melted['Rank'], errors='coerce')
melted.dropna(subset=['Rank'], inplace=True)

pivoted = melted.pivot_table(index=['Country', 'App'], columns='Ranking', values='Rank').reset_index()
pivoted.rename(columns={'Top Free Median': 'Free_Rank', 'Top Grossing Median': 'Gross_Rank'}, inplace=True)

country_stats = pivoted.groupby('Country').agg({'Free_Rank': 'median', 'Gross_Rank': 'median'}).reset_index()
country_stats['Log_Free_Rank'] = np.log10(country_stats['Free_Rank'] + 1)
country_stats['Log_Gross_Rank'] = np.log10(country_stats['Gross_Rank'] + 1)

X = country_stats[['Log_Free_Rank', 'Log_Gross_Rank']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
country_stats['Cluster'] = kmeans.fit_predict(X_scaled)

# 1-Index the clusters
country_stats['User_Cluster'] = country_stats['Cluster'] + 1

# 2. App Filtering

app_perf = pivoted.merge(country_stats[['Country', 'User_Cluster']], on='Country', how='inner')
app_perf_no_duo = app_perf[app_perf['App'] != 'Duolingo']

cluster_app_perf = app_perf_no_duo.groupby(['User_Cluster', 'App']).agg({
    'Gross_Rank': 'median',
    'Free_Rank': 'median'
}).reset_index()


# 3. Exact Colour & Label Mapping

cluster_colors = {
    1: '#d62728', # Red (Germany)
    2: '#1f77b4', # Blue (Japan)
    3: '#2ca02c', # Green (Argentina)
    4: '#9467bd'  # Purple (Poland)
}

cluster_labels = {
    1: "Cluster 1: Saturated Markets (e.g., Germany)",
    2: "Cluster 2: The Dead Zone (e.g., Japan)",
    3: "Cluster 3: High-Intent Niche (e.g., Argentina)",
    4: "Cluster 4: The Golden Quadrant (e.g., Poland)"
}

# ==========================================
# 4. GRAPH 1: TOP GROSSING (MONETIZATION)
# ==========================================
fig_g, axes_g = plt.subplots(2, 2, figsize=(16, 12))
fig_g.suptitle('Secondary Market Leaders by Monetization (Excluding Duolingo)', fontsize=20, fontweight='bold')
axes_g = axes_g.flatten()

for i, c in enumerate([1, 2, 3, 4]):
    subset = cluster_app_perf[cluster_app_perf['User_Cluster'] == c].sort_values('Gross_Rank').head(5)

    # Calculate max value to dynamically set boundaries
    max_val = subset['Gross_Rank'].max()

    sns.barplot(data=subset, x='Gross_Rank', y='App', ax=axes_g[i], color=cluster_colors[c])
    axes_g[i].set_title(cluster_labels[c], fontsize=14, color=cluster_colors[c], fontweight='bold')
    axes_g[i].set_xlabel('Median Gross Rank (Lower = Better Revenue)', fontsize=12)
    axes_g[i].set_ylabel('')

    # Add 25% empty space on the left side of the inverted axis so text doesn't overlap
    axes_g[i].set_xlim(max_val * 1.25, 0)

    for p in axes_g[i].patches:
        width = p.get_width()
        padding = max_val * 0.02 # Dynamic padding based on the specific chart's scale
        axes_g[i].text(width + padding, p.get_y() + p.get_height()/2, f'#{int(width)}', ha='right', va='center')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
fig_g.savefig('competitors_grossing.png', dpi=300)

# ==========================================
# 5. GRAPH 2: TOP FREE (USAGE VOLUME)
# ==========================================
fig_f, axes_f = plt.subplots(2, 2, figsize=(16, 12))
fig_f.suptitle('Secondary Market Leaders by Usage Volume (Excluding Duolingo)', fontsize=20, fontweight='bold')
axes_f = axes_f.flatten()

for i, c in enumerate([1, 2, 3, 4]):
    subset = cluster_app_perf[cluster_app_perf['User_Cluster'] == c].sort_values('Free_Rank').head(5)

    # Calculate max value to dynamically set boundaries
    max_val = subset['Free_Rank'].max()

    sns.barplot(data=subset, x='Free_Rank', y='App', ax=axes_f[i], color=cluster_colors[c])
    axes_f[i].set_title(cluster_labels[c], fontsize=14, color=cluster_colors[c], fontweight='bold')
    axes_f[i].set_xlabel('Median Free Rank (Lower = More Downloads)', fontsize=12)
    axes_f[i].set_ylabel('')

    # Add 25% empty space on the left side of the inverted axis
    axes_f[i].set_xlim(max_val * 1.25, 0)

    for p in axes_f[i].patches:
        width = p.get_width()
        padding = max_val * 0.02 # Dynamic padding
        axes_f[i].text(width + padding, p.get_y() + p.get_height()/2, f'#{int(width)}', ha='right', va='center')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
fig_f.savefig('competitors_free.png', dpi=300)
print("Charts successfully generated with dynamic margins to prevent text overlap.")




K-Means Validation Scores:
2	44.89
3	29.34
4	19.43
5	15.97
6	13.00
7	10.44
8	8.18
9	6.62
10	5.91
